# 82 Boxes: Checking the Processed Tables
Reads the five tables saved by `src/clean_data.py` in `data/processed/`. The automatic checks already ran inside that script (see `logs/clean_data.log`). This notebook is the human check on the final output: look at the rows, then compare a few numbers against ESPN.

A match with ESPN shows two sources agree. It doesn't prove either one is absolutely correct.

## 0. Load the tables

In [1]:
import pandas as pd

pd.set_option("display.max_columns", None)

P = "../data/processed/"
teams = pd.read_parquet(P + "teams.parquet")
games = pd.read_parquet(P + "games.parquet")
players = pd.read_parquet(P + "players.parquet")
pgs = pd.read_parquet(P + "player_game_stats.parquet")
shots = pd.read_parquet(P + "shots.parquet")

for name, df in [("teams", teams), ("games", games), ("players", players),
                 ("player_game_stats", pgs), ("shots", shots)]:
    print(f"{name:20s} {df.shape}")
# expect: teams 30, games 1230, players 582, player_game_stats 26651, shots 219160

teams                (30, 5)
games                (1230, 7)
players              (582, 2)
player_game_stats    (26651, 19)
shots                (219160, 16)


## 1. A look at each table

In [2]:
games.head()

,game_id,season,game_type,game_date,home_team_id,away_team_id,neutral_site
0,0022501188,2025,Regular Season,2026-04-12,1610612754,1610612765,False
1,0022501193,2025,Regular Season,2026-04-12,1610612742,1610612741,False
2,0022501194,2025,Regular Season,2026-04-12,1610612745,1610612763,False
3,0022501200,2025,Regular Season,2026-04-12,1610612757,1610612758,False
4,0022501198,2025,Regular Season,2026-04-12,1610612747,1610612762,False


In [3]:
pgs.head()

,game_id,player_id,team_id,min,pts,fgm,fga,fg3m,fg3a,ftm,fta,oreb,dreb,ast,stl,blk,tov,pf,plus_minus
0,0022501193,1642948,1610612742,38.28,15,6,11,1,2,2,2,2,7,23,2,0,4,2,23
1,0022501195,1642852,1610612740,41.05,30,11,28,1,2,7,8,7,15,4,1,1,4,5,-5
2,0022501192,1630567,1610612761,31.99,18,8,11,0,0,2,2,0,12,12,1,3,1,0,35
3,0022501195,1642847,1610612740,41.38,36,12,28,0,5,12,14,3,7,5,2,1,2,3,0
4,0022501195,1642866,1610612750,31.27,24,9,12,0,0,6,9,6,7,2,0,7,0,3,-1


In [12]:
shots.shape
shots.head()


,game_id,game_event_id,player_id,team_id,period,minutes_remaining,seconds_remaining,action_type,shot_type,shot_zone_basic,shot_zone_area,shot_zone_range,shot_distance,loc_x,loc_y,shot_made_flag
0,0022500007,9,1641709,1610612765,1,11,28,Layup Shot,2PT Field Goal,In The Paint (Non-RA),Right Side(R),8-16 ft.,8,64,51,0
1,0022500007,11,1641709,1610612765,1,11,25,Tip Layup Shot,2PT Field Goal,Restricted Area,Center(C),Less Than 8 ft.,0,0,0,1
2,0022500007,16,202699,1610612765,1,11,4,Jump Shot,3PT Field Goal,Above the Break 3,Right Side Center(RC),24+ ft.,25,140,216,1
3,0022500007,21,1629130,1610612765,1,10,34,Driving Floating Jump Shot,2PT Field Goal,In The Paint (Non-RA),Center(C),8-16 ft.,9,22,88,0
4,0022500007,32,1629130,1610612765,1,9,56,Jump Shot,3PT Field Goal,Above the Break 3,Right Side Center(RC),24+ ft.,26,148,215,1


## 2. Team scores from player rows
Team scores are not stored. They come from adding up each team's player points. Here is each game's score and point differential, which is what the team heatmap will use.

In [5]:
team_pts = pgs.groupby(["game_id", "team_id"])["pts"].sum().reset_index()

home = team_pts.rename(columns={"team_id": "home_team_id", "pts": "home_pts"})
away = team_pts.rename(columns={"team_id": "away_team_id", "pts": "away_pts"})

scores = (games.merge(home, on=["game_id", "home_team_id"])
               .merge(away, on=["game_id", "away_team_id"]))
scores["point_diff"] = scores["home_pts"] - scores["away_pts"]  # positive = home team won

print(scores.shape)  # 1230 rows, one per game
scores[["game_id", "game_date", "home_team_id", "away_team_id", "home_pts", "away_pts", "point_diff"]].head()

(1230, 10)


,game_id,game_date,home_team_id,away_team_id,home_pts,away_pts,point_diff
0,0022501188,2026-04-12,1610612754,1610612765,121,133,-12
1,0022501193,2026-04-12,1610612742,1610612741,149,128,21
2,0022501194,2026-04-12,1610612745,1610612763,132,101,31
3,0022501200,2026-04-12,1610612757,1610612758,122,110,12
4,0022501198,2026-04-12,1610612747,1610612762,131,107,24


## 3. One team's season: the Nets (team id 1610612751)
The Nets should have 82 games. Their win-loss record comes from the scores above. Compare it with the Nets' regular season record on ESPN.

In [6]:
nets = 1610612751

nets_games = scores[(scores["home_team_id"] == nets) | (scores["away_team_id"] == nets)]
print("Games:", len(nets_games))  # 82

nets_won = (((nets_games["home_team_id"] == nets) & (nets_games["point_diff"] > 0))
            | ((nets_games["away_team_id"] == nets) & (nets_games["point_diff"] < 0)))
print("Record:", nets_won.sum(), "-", (~nets_won).sum())  # compare with ESPN

Games: 82
Record: 20 - 62


## 4. Spot-check one player against ESPN: Egor Demin (player id 1642856)
ESPN lists 52 GP, 179-449 FG, 124-322 3P and 536 points.

In [7]:
demin = pgs[pgs["player_id"] == 1642856]

print("GP :", len(demin))                                     # ESPN: 52
print("FG :", demin["fgm"].sum(), "-", demin["fga"].sum())    # ESPN: 179-449
print("3P :", demin["fg3m"].sum(), "-", demin["fg3a"].sum())  # ESPN: 124-322
print("PTS:", demin["pts"].sum())                             # ESPN: 536

GP : 52
FG : 179 - 449
3P : 124 - 322
PTS: 536


## 5. Hand-check one game
Pick a game, open its box score on ESPN, and compare by eye. Change `game_id` to try another game.

In [8]:
game_id = "0022501192"  # TOR vs BKN on 2026-04-12

box = (pgs[pgs["game_id"] == game_id]
       .merge(players, on="player_id")
       .merge(teams[["team_id", "abbreviation"]], on="team_id"))
box[["abbreviation", "player_name", "min", "pts", "fgm", "fga", "fg3m", "fg3a", "ftm", "fta",
     "oreb", "dreb", "ast", "stl", "blk", "tov", "pf"]]

,abbreviation,player_name,min,pts,fgm,fga,fg3m,fg3a,ftm,fta,oreb,dreb,ast,stl,blk,tov,pf
0,TOR,Scottie Barnes,31.99,18,8,11,0,0,2,2,0,12,12,1,3,1,0
1,TOR,Brandon Ingram,32.00,25,7,10,3,5,8,8,1,8,2,0,1,1,1
2,BKN,Chaney Johnson,36.33,16,6,14,0,3,4,5,5,8,2,1,0,1,3
3,BKN,Tyson Etienne,36.75,20,6,16,2,10,6,8,0,1,4,2,0,1,2
4,TOR,RJ Barrett,28.62,26,9,15,2,5,6,9,1,2,2,0,0,1,5
5,BKN,Ben Saraf,26.37,15,6,10,0,1,3,4,2,1,4,2,0,4,6
6,TOR,Jakob Poeltl,19.25,11,5,5,0,0,1,5,0,5,2,2,1,3,2
7,BKN,E.J. Liddell,27.38,17,5,10,3,5,4,4,1,3,1,1,0,1,6
8,BKN,Jalen Wilson,26.35,9,3,9,0,0,3,6,3,5,3,1,0,2,5
9,BKN,Malachi Smith,35.52,8,3,9,2,6,0,0,1,4,6,0,0,2,3


In [9]:
# Team totals for that game, added up from the player rows above. Compare with the final score on ESPN.
box.groupby("abbreviation")[["pts", "fgm", "fga", "fg3m", "fg3a", "ftm", "fta", "oreb", "dreb", "ast", "stl", "blk", "tov", "pf"]].sum()

,pts,fgm,fga,fg3m,fg3a,ftm,fta,oreb,dreb,ast,stl,blk,tov,pf
abbreviation,,,,,,,,,,,,,,
BKN,101,35,86,9,34,22,29,13,25,21,9,0,15,27
TOR,136,51,80,12,27,22,29,3,40,36,9,8,10,20
